In [21]:
# %%

IGNORE_DIRS = {
    ".git",
    ".github",
    ".idea",
    ".vscode",
    "__pycache__",
    "node_modules",
    "vendor",
    "dist",
    "build",
    ".next",
    ".nuxt",
    ".cache",
    ".venv",
    "venv",
    "env",
    "coverage",
    ".pytest_cache",
    ".mypy_cache",
}

IGNORE_FILES = {
    ".gitignore",
    ".gitattributes",
    ".gitmodules",
    ".DS_Store",
    "Thumbs.db",
    "package-lock.json",
    "yarn.lock",
    "pnpm-lock.yaml",
    "poetry.lock",
    "Pipfile.lock",
    "Cargo.lock",
    "composer.lock",
    "Gemfile.lock",
}

IGNORE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".gif",
    ".webp",
    ".ico",
    ".svg",
    ".pdf",
    ".zip",
    ".tar",
    ".gz",
    ".7z",
    ".exe",
    ".dll",
    ".so",
    ".dylib",
    ".class",
    ".jar",
    ".pyc",
    ".pyo",
}

SOURCE_EXTENSIONS = {
    ".py",
    ".js",
    ".ts",
    ".tsx",
    ".jsx",
    ".java",
    ".go",
    ".rs",
    ".cpp",
    ".c",
    ".h",
    ".hpp",
    ".cs",
    ".php",
    ".rb",
    ".swift",
    ".kt",
    ".scala",
    ".sql",
    ".html",
    ".css",
    ".scss",
    ".json",
    ".yaml",
    ".yml",
    ".toml",
    ".xml",
    ".md",
    ".sh",
    ".dockerfile",
}

# %%
from pathlib import Path
import os

MAX_FILE_SIZE = 1_000_000  


def walk_repository(root: Path):
    root = Path(root)

    for current_root, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in IGNORE_DIRS]

        for file in files:
            path = Path(current_root) / file

            if file in IGNORE_FILES:
                continue

            if path.suffix.lower() not in SOURCE_EXTENSIONS:
                continue

            if path.suffix.lower() in IGNORE_EXTENSIONS:
                continue

            if path.stat().st_size > MAX_FILE_SIZE:
                continue

            yield path

# %%
from pathlib import Path

def print_repository_files(root):
    count = 0

    for path in walk_repository(Path(root)):
        print(path)
        count += 1

    print(f"\nTotal files found: {count}")

# %%
print_repository_files("../../backend/src/chunking")

# %%





..\..\backend\src\chunking\chunker.py
..\..\backend\src\chunking\config.py
..\..\backend\src\chunking\hash_chunk.py
..\..\backend\src\chunking\metadata.py
..\..\backend\src\chunking\models.py
..\..\backend\src\chunking\parser_factory.py
..\..\backend\src\chunking\__init__.py

Total files found: 7


In [22]:
CHUNK_NODE_TYPES = {
    "python": {
        "class_definition",
        "function_definition",
    },
    "javascript": {
        "class_declaration",
        "function_declaration",
        "method_definition",
        "generator_function_declaration",
    },
    "typescript": {
        "class_declaration",
        "function_declaration",
        "method_definition",
        "generator_function_declaration",
        "interface_declaration",
    },
    "java": {
        "class_declaration",
        "interface_declaration",
        "enum_declaration",
        "method_declaration",
        "constructor_declaration",
    },
    "go": {
        "function_declaration",
        "method_declaration",
        "type_declaration",
    },
}

CLASS_NODE_TYPES = {
    "python": {"class_definition"},
    "javascript": {"class_declaration"},
    "typescript": {"class_declaration"},
    "java": {
        "class_declaration",
        "interface_declaration",
        "enum_declaration",
    },
    "go": {"type_declaration"},
}

In [23]:
from dataclasses import dataclass
from typing import Any


@dataclass(slots=True)
class Chunk:
    text: str
    metadata: dict[str, Any]

In [24]:
from tree_sitter import Language, Parser

import tree_sitter_python
import tree_sitter_javascript
import tree_sitter_typescript
import tree_sitter_java
import tree_sitter_go


LANGUAGES = {
    "python": Language(tree_sitter_python.language()),
    "javascript": Language(tree_sitter_javascript.language()),
    "typescript": Language(tree_sitter_typescript.language_typescript()),
    "java": Language(tree_sitter_java.language()),
    "go": Language(tree_sitter_go.language()),
}


class ParserFactory:
    def __init__(self):
        self._cache = {}

    def get(self, language: str) -> Parser:
        if language not in LANGUAGES:
            raise ValueError(f"Unsupported language: {language}")

        if language not in self._cache:
            self._cache[language] = Parser(LANGUAGES[language])

        return self._cache[language]

In [25]:
import hashlib


def build_metadata(
    *,
    file_path,
    language,
    node,
    symbol,
    parent,
    node_type,
    text,
):
    return {
        "file_path": file_path,
        "language": language,
        "symbol": symbol,
        "parent": parent,
        "node_type": node_type,
        "start_line": node.start_point[0] + 1,
        "end_line": node.end_point[0] + 1,
        "start_byte": node.start_byte,
        "end_byte": node.end_byte,
        "hash": hashlib.sha256(text.encode()).hexdigest(),
    }

In [26]:
from enum import Enum


class Language(str, Enum):
    PYTHON = "python"
    JAVASCRIPT = "javascript"
    TYPESCRIPT = "typescript"
    JAVA = "java"
    GO = "go"
    RUST = "rust"
    CPP = "cpp"
    C = "c"
    CSHARP = "csharp"
    PHP = "php"
    RUBY = "ruby"
    SWIFT = "swift"
    KOTLIN = "kotlin"

    HTML = "html"
    CSS = "css"
    SCSS = "scss"

    JSON = "json"
    YAML = "yaml"
    XML = "xml"
    TOML = "toml"

    SQL = "sql"

    MARKDOWN = "markdown"
    TEXT = "text"

    UNKNOWN = "unknown"


EXTENSION_MAP = {
    # Python
    ".py": Language.PYTHON,
    ".pyi": Language.PYTHON,

    # JavaScript
    ".js": Language.JAVASCRIPT,
    ".jsx": Language.JAVASCRIPT,
    ".mjs": Language.JAVASCRIPT,
    ".cjs": Language.JAVASCRIPT,

    # TypeScript
    ".ts": Language.TYPESCRIPT,
    ".tsx": Language.TYPESCRIPT,

    # Java
    ".java": Language.JAVA,

    # Go
    ".go": Language.GO,

    # Rust
    ".rs": Language.RUST,

    # C
    ".c": Language.C,
    ".h": Language.C,

    # C++
    ".cpp": Language.CPP,
    ".cc": Language.CPP,
    ".cxx": Language.CPP,
    ".hpp": Language.CPP,
    ".hh": Language.CPP,

    # C#
    ".cs": Language.CSHARP,

    # PHP
    ".php": Language.PHP,

    # Ruby
    ".rb": Language.RUBY,

    # Swift
    ".swift": Language.SWIFT,

    # Kotlin
    ".kt": Language.KOTLIN,
    ".kts": Language.KOTLIN,

    # Web
    ".html": Language.HTML,
    ".htm": Language.HTML,
    ".css": Language.CSS,
    ".scss": Language.SCSS,

    # Config
    ".json": Language.JSON,
    ".yaml": Language.YAML,
    ".yml": Language.YAML,
    ".xml": Language.XML,
    ".toml": Language.TOML,

    # Database
    ".sql": Language.SQL,

    # Documentation
    ".md": Language.MARKDOWN,
    ".mdx": Language.MARKDOWN,
    ".txt": Language.TEXT,
}


SPECIAL_FILES = {
    "Dockerfile": Language.TEXT,
    "Makefile": Language.TEXT,
    "CMakeLists.txt": Language.TEXT,

    ".gitignore": Language.TEXT,
    ".dockerignore": Language.TEXT,
    ".editorconfig": Language.TEXT,
    ".env": Language.TEXT,

    "package.json": Language.JSON,
    "tsconfig.json": Language.JSON,
    "pyproject.toml": Language.TOML,
    "requirements.txt": Language.TEXT,
}


SHEBANG_MAP = {
    "python": Language.PYTHON,
    "python3": Language.PYTHON,
    "node": Language.JAVASCRIPT,
    "ruby": Language.RUBY,
    "bash": Language.TEXT,
    "sh": Language.TEXT,
    "zsh": Language.TEXT,
}

In [27]:
from pathlib import Path

def detect_language(path: str | Path) -> Language:
    """
    Detect the programming language of a file.
    Detection order:
    1. Special filenames (Dockerfile, Makefile, etc.)
    2. File extension
    3. Shebang (#!/usr/bin/env python)
    """

    path = Path(path)

    if path.name in SPECIAL_FILES:
        return SPECIAL_FILES[path.name]

    language = EXTENSION_MAP.get(path.suffix.lower())
    if language:
        return language

    try:
        with path.open("r", encoding="utf-8", errors="ignore") as f:
            first_line = f.readline().strip()

        if first_line.startswith("#!"):
            for interpreter, language in SHEBANG_MAP.items():
                if interpreter in first_line:
                    return language
    except OSError:
        pass

    return Language.UNKNOWN

In [28]:
import hashlib

def normalize_for_hash(text: str) -> str:
    return " ".join(text.split())

def dedup_stream(chunks_iter):
    seen = set()
    for c in chunks_iter:
        h = hashlib.sha256(normalize_for_hash(c.text).encode("utf8")).hexdigest()
        if h in seen:
            continue
        seen.add(h)
        c.metadata["content_hash"] = h
        yield c

In [29]:

class ASTChunker:
    def __init__(self):
        self.factory = ParserFactory()

    def chunk_file(self, source_code: str, language: str, file_path: str):
        if language not in CHUNK_NODE_TYPES:
            return

        parser = self.factory.get(language)
        source = source_code.encode("utf8")
        tree = parser.parse(source)

        chunk_nodes = CHUNK_NODE_TYPES[language]
        class_nodes = CLASS_NODE_TYPES[language]

        def node_name(node):
            name = node.child_by_field_name("name")
            return source[name.start_byte:name.end_byte].decode() if name else None

        def visit(node, parent=None):
            if node.type in chunk_nodes:
                text = source[node.start_byte:node.end_byte].decode()
                symbol = node_name(node)
                yield Chunk(
                    text=text,
                    metadata=build_metadata(
                        file_path=file_path, language=language, node=node,
                        symbol=symbol, parent=parent, node_type=node.type, text=text,
                    )
                )
                next_parent = symbol if node.type in class_nodes else parent
                for child in node.children:
                    yield from visit(child, next_parent)
                return

            for child in node.children:
                yield from visit(child, parent)

        yield from visit(tree.root_node)
        yield from self.extract_module_level(tree.root_node, source, chunk_nodes, file_path, language)

    def extract_module_level(
        self,
        root,
        source,
        chunk_nodes,
        file_path,
        language,
    ):
        chunks = []

        cursor = 0

        for child in root.children:
            if child.type not in chunk_nodes:
                continue

            if child.start_byte > cursor:
                text = source[
                    cursor:child.start_byte
                ].decode().strip()

                if text:
                    chunks.append(
                        Chunk(
                            text=text,
                            metadata={
                                "file_path": file_path,
                                "language": language,
                                "node_type": "module_level",
                                "symbol": None,
                                "parent": None,
                            }
                        )
                    )

            cursor = child.end_byte

        if cursor < len(source):
            text = source[cursor:].decode().strip()

            if text:
                chunks.append(
                   Chunk(
                       text=text,
                       metadata={
                           "file_path": file_path,
                           "language": language,
                           "node_type": "module_level",
                           "symbol": None,
                           "parent": None,
                       }
                   )
                )

        return chunks

In [35]:
chunker = ASTChunker()
all_chunks = []
for file_path in walk_repository("workspace/repos/0e17c42e4f83f98c/"):
    lang = detect_language(file_path)

    if lang.value == "unknown":
        continue
    try:
        source = file_path.read_text(encoding="utf-8", errors="ignore")
    except OSError:
        continue
    all_chunks.extend(chunker.chunk_file(
        source, lang.value, str(file_path)))

deduped = list(dedup_stream(iter(all_chunks)))


In [36]:
deduped

[Chunk(text='class AudioGenerationBase(BaseModel):\n    audio_duration: float = 180.0\n    seed: int = -1\n    guidance_scale: float = 15.0\n    infer_step: int = 60\n    instrumental: bool = False', metadata={'file_path': 'workspace\\repos\\0e17c42e4f83f98c\\backend\\main.py', 'language': 'python', 'symbol': 'AudioGenerationBase', 'parent': None, 'node_type': 'class_definition', 'start_line': 31, 'end_line': 36, 'start_byte': 801, 'end_byte': 980, 'hash': '990f33ac9b5bd32df982c0c769d6a6cf45c15353c526d1a8ad9d980ed4d26e85', 'content_hash': '24ced81e9d4c6921244a14a9242d7e757c4e81651e616d0a976f8c3a4d8d9126'}),
 Chunk(text='class GenerateFromDescriptionRequest(AudioGenerationBase):\n    full_described_song: str', metadata={'file_path': 'workspace\\repos\\0e17c42e4f83f98c\\backend\\main.py', 'language': 'python', 'symbol': 'GenerateFromDescriptionRequest', 'parent': None, 'node_type': 'class_definition', 'start_line': 39, 'end_line': 40, 'start_byte': 983, 'end_byte': 1070, 'hash': '08e14df